In [79]:
import wrds
import pandas as pd
db = wrds.Connection(wrds_username=username)

ticker = input("Please input three companies，separated by commas（e.g.AAPL,NKE,BIRD）: ").split(",")
company_dict = {
    ticker[0].strip(): "Company A",
    ticker[1].strip(): "Company B",
    ticker[2].strip(): "Company C"
}
start_date = "2024.01.01"

stock_tuple = tuple(company_dict.keys())

selected_columns = """
b.htsymbol,
b.permno,
a.date,
a.prc,
a.ret
"""

sql_query = f"""
SELECT {selected_columns}
FROM crsp.msf AS a
LEFT JOIN crsp.msfhdr AS b
    ON a.permno = b.permno
WHERE b.htsymbol IN {stock_tuple}
AND a.date >= '{start_date}'
"""
stock_joined = db.raw_sql(sql_query, date_cols=["date"])

stock_joined_renamed = stock_joined.rename(columns={
    "htsymbol": "name",
    "prc": "price",
    "ret": "monthly_return",
})
stock_joined_sorted = stock_joined_renamed.sort_values(
    by=["name", "date"]
).reset_index(drop=True)

stock_joined_sorted["year"] = stock_joined_sorted["date"].dt.year

stock_by_return = stock_joined_sorted[stock_joined_sorted["monthly_return"].notna()].sort_values(by="monthly_return", ascending=False)

ai_table = stock_joined_sorted[["name", "date", "monthly_return", "year","price"]].head(40)
display(ai_table)

try:
    ai_table.to_excel("stock_ai_table.xlsx", index=False)

    print("Excel Export successful")
    print("Generated file：")

except Exception as e:
    print("Excel Export fail：", e)

db.close()
print("The database connection has been closed!")

Loading library list...
Done


Please input three companies，separated by commas（e.g.AAPL,NKE,BIRD）:  AAPL,NKE,BIRD


,name,date,monthly_return,year,price
0,AAPL,2024-01-31,-0.042227,2024,184.39999
1,AAPL,2024-02-29,-0.018492,2024,180.75
2,AAPL,2024-03-28,-0.051286,2024,171.48
3,AAPL,2024-04-30,-0.006706,2024,170.33
4,AAPL,2024-05-31,0.130159,2024,192.25
5,AAPL,2024-06-28,0.095553,2024,210.62
6,AAPL,2024-07-31,0.054411,2024,222.08
7,AAPL,2024-08-30,0.032286,2024,229.0
8,AAPL,2024-09-30,0.017467,2024,233.0
9,AAPL,2024-10-31,-0.030429,2024,225.91


Excel Export successful
Generated file：
The database connection has been closed!
